# The credit corpus: what was delivered

Start here. We describe the 25 registered datasets before learning anything about model performance. PD and LGD are separate tasks: 17 default-classification tables and 8 loss-given-default tables.

Read in order: availability → shape and missingness → row concentration → provenance → input checks. The next notebook follows these files through preprocessing. Raw-file measurements deliberately precede dataset-specific corrections; a missing registered target name can therefore be an expected rename rather than a missing label.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from IPython.display import display
from src.visualize import style
from src.visualize.figures import FigureSaver
from src.visualize import campaign as cp
from src.data.dataset_names import display_frame
from src.visualize.inputs import analysis_root
style.apply()
sink = FigureSaver('00_general/01_raw_data')
report = cp.NotebookReport('The credit corpus: what was delivered')
from src.visualize import corpus as cv


## 1. Availability and inventory

Every registered dataset remains in the inventory, including missing files. Public names and private aliases are used throughout.

In [ ]:
raw = cv.raw_inventory()
display(raw[['track','dataset_id','rows','features','file_mb','target_in_raw']])
report.add('1. Availability and inventory', cv.inventory_summary(raw, 'Raw'))

## 2. Corpus geometry

Logarithmic axes make small and large tables visible together. The panels separate tasks; no row-weighted model conclusion is drawn from corpus size.

In [ ]:
cp.show(sink, cv.plot_geometry(raw, 'Raw'))
report.add('2. Corpus geometry', raw.groupby('track')[['rows','features']].agg(['min','median','max']).to_string())

## 3. Missingness without hiding datasets

Pages contain at most 12 dataset labels. Raw missingness counts all delivered cells, including the target when present. Unknown files are blank, not zero.

In [ ]:
cp.show(sink, cv.plot_profiles(raw, 'Raw'))
report.add('3. Missingness', raw.groupby('track').missing.agg(['count','mean','max']).to_string())

## 4. How unequal are the table sizes?

A small number of large tables can dominate row-weighted training. This curve motivates comparing table visitation and full-pass sampling; it is not an estimate of the actual optimizer weights.

In [ ]:
cp.show(sink, cv.plot_concentration(raw))
report.add('4. Row concentration', raw.groupby('track').rows.agg(['sum','median','max']).to_string())

## 5. Source coverage

Provenance categories describe where the tables came from. They do not establish independence, licensing status or absence from base-model pretraining.

In [ ]:
cp.show(sink, cv.plot_sources(raw))
report.add('5. Source coverage', raw.groupby(['source','track']).size().to_string())

## 6. Delivery checks

These flags identify inputs to inspect. Do not interpret a raw target-name mismatch before checking the dataset-specific preprocessing rule.

In [ ]:
flags = raw[raw.rows.isna() | ~raw.target_in_raw]
display(flags[['track','dataset_id','target_in_raw']])
report.add('6. Delivery checks', flags[['track','dataset_id','target_in_raw']].to_string(index=False) if len(flags) else 'All registered files and target names are present.')

## Summary

The following text repeats the sections in order. It is included verbatim in `All_Results.md`.

In [ ]:
print(report.summary(sink))